# Setup

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git accelerate huggingface_hub

In [ ]:
!pip install requests beautifulsoup4 lxml

In [ ]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Translation Agent

## Commentary Parser

In [ ]:
import re
import requests
from bs4 import BeautifulSoup

def parse_enduring_word(book: str, chapter: int, start_verse: int, end_verse: int):
    """Scrapes Enduring Word commentary, isolates target boundaries,

    and returns a clean list of structured blocks (No console prints).
    """
    book_url = book.lower().replace(" ", "-")
    url = f"https://enduringword.com/bible-commentary/{book_url}-{chapter}/"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"❌ HTTP Error {response.status_code}")
        return []

    soup = BeautifulSoup(response.content, "lxml")
    content_area = soup.find("div", class_="entry-content") or soup.find(
        "article"
    )
    if not content_area:
        print("❌ Could not find the core article content area.")
        return []

    current_verse_context = None
    structured_blocks = []
    pending_h3 = None

    for element in content_area.find_all(["h3", "h4", "p"]):
        # 1. Skip the main Bible text blocks entirely
        if (
            element.name == "p"
            and element.has_attr("class")
            and "ew-bible-text" in element["class"]
        ):
            continue

        text = element.get_text().strip()
        if not text:
            continue

        # 2. --- MAIN SECTION HEADERS (H3) ---
        if element.name == "h3" and not ("(" in text and ")" in text):
            pending_h3 = text
            continue

        # 3. --- SCRIPTURE ANCHORS (H4 / H3 with parentheses) ---
        if element.name == "h4" or (
            element.name == "h3" and "(" in text and ")" in text
        ):
            verse_match = re.search(
                r"\((\d+)[a-zA-Z]?(?:\s*-\s*(\d+)[a-zA-Z]?)?\)", text
            )
            if verse_match:
                v_start = int(verse_match.group(1))
                v_end = (
                    int(verse_match.group(2)) if verse_match.group(2) else v_start
                )

                current_verse_context = (v_start, v_end)

                # Lower limit boundary break
                if v_start > end_verse:
                    break

                # Append matching section layout headings
                if v_end >= start_verse and v_start <= end_verse:
                    if pending_h3:
                        structured_blocks.append(
                            {
                                "verse_range": current_verse_context,
                                "type": "main_section_header",
                                "bold_quote": "",
                                "full_text": pending_h3,
                            }
                        )
                        pending_h3 = None

                    structured_blocks.append(
                        {
                            "verse_range": current_verse_context,
                            "type": "scripture_anchor",
                            "bold_quote": "",
                            "full_text": text,
                        }
                    )
                continue

        # 4. --- PARSE BODY PARAGRAPHS & OUTLINE HOOKS (p) ---
        if element.name == "p":
            if re.search(r"©1996.*Enduring\s+Word\s+Bible\s+Commentary", text):
                continue

            bold_tag = element.find("strong")
            bold_quote = bold_tag.get_text().strip() if bold_tag else ""

            if bold_quote and bold_quote == text:
                continue

            if (
                current_verse_context is None
                or current_verse_context[1] < start_verse
            ):
                continue

            v_start, v_end = current_verse_context
            if v_start > end_verse:
                break

            structured_blocks.append(
                {
                    "verse_range": current_verse_context,
                    "type": "body_paragraph",
                    "bold_quote": bold_quote,
                    "full_text": text,
                }
            )

    return structured_blocks


def print_commentary_cache(structured_blocks: list):
    """Takes a local cache array of structured blocks and prints them

    with beautiful hierarchical tree indentation.
    """
    if not structured_blocks:
        print("⚠️ Local cache block array is empty.")
        return

    print(
        f"🖥️ Rendering {len(structured_blocks)} structured elements from local memory cache:\n"
    )

    for block in structured_blocks:
        text = block["full_text"]
        b_type = block["type"]

        if b_type == "main_section_header":
            print(f"\n🟢 [H3] {text}")

        elif b_type == "scripture_anchor":
            print(f"  🔵 [H4 Anchor] {text}")

        elif b_type == "body_paragraph":
            # Identify line-nesting structures for formatting lookups
            is_roman_numeral = re.match(
                r"^(v|i|x)+\.", text, re.IGNORECASE
            )  # i., ii., etc.
            is_alpha_letter = re.match(r"^[a-z]\.", text)  # a., b., c., etc.

            if is_roman_numeral:
                print(f"      🔹 [Sub-point] {text}")
            elif is_alpha_letter:
                print(f"    🔸 [Point] {text}")
            else:
                print(f"    ... [Standard Body] {text}")

## Bible Parser

In [ ]:
BOOK_CODE_MAP = {
    "genesis": "sa",
    "exodus": "xu",
    "leviticus": "le",
    "numbers": "dan",
    "deuteronomy": "phu",
    "joshua": "gios",
    "judges": "cac",
    "ruth": "ru",
    "1 samuel": "1sa",
    "2 samuel": "2sa",
    "1 kings": "1vua",
    "2 kings": "2vua",
    "1 chronicles": "1su",
    "2 chronicles": "2su",
    "ezra": "exo",
    "nehemiah": "ne",
    "esther": "et",
    "job": "giop",
    "psalms": "thi",
    "proverbs": "ch",
    "ecclesiastes": "tr",
    "song of solomon": "nha",
    "isaiah": "es",
    "jeremiah": "gie",
    "lamentations": "ca",
    "ezekiel": "exe",
    "daniel": "da",
    "hosea": "os",
    "joel": "gio",
    "amos": "am",
    "obadiah": "ap",
    "jonah": "gion",
    "micah": "mi",
    "nahum": "na",
    "habakkuk": "ha",
    "zephaniah": "so",
    "haggai": "ag",
    "zechariah": "xa",
    "malachi": "ma",

    "matthew": "mat",
    "mark": "mac",
    "luke": "lu",
    "john": "gi",
    "acts": "cong",
    "romans": "ro",
    "1 corinthians": "1co",
    "2 corinthians": "2co",
    "galatians": "ga",
    "ephesians": "eph",
    "philippians": "phi",
    "colossians": "co",
    "1 thessalonians": "1te",
    "2 thessalonians": "2te",
    "1 timothy": "1ti",
    "2 timothy": "2ti",
    "titus": "tit",
    "philemon": "phil",
    "hebrews": "he",
    "james": "gia",
    "1 peter": "1phi",
    "2 peter": "2phi",
    "1 john": "1gi",
    "2 john": "2gi",
    "3 john": "3gi",
    "jude": "giu",
    "revelation": "kh"
}

VI_TO_EN_BOOKS = {
    "Ê-SAI": "Isaiah",

    "MA-THI-Ơ": "Matthew", "MÁC": "Mark", "LU-CA": "Luke", "GIĂNG": "John",
    "CÔNG-VỤ": "Acts", "RÔ-MA": "Romans", "I CÔ-RINH-TÔ": "1 Corinthians",
    "II CÔ-RINH-TÔ": "2 Corinthians", "GA-LA-TI": "Galatians", "Ê-PHÊ-SÔ": "Ephesians",
    "PHI-LÍP": "Philippians", "CÔ-LÔ-SE": "Colossians", "I TÊ-SA-LÔ-NI-CA": "1 Thessalonians",
    "II TÊ-SA-LÔ-NI-CA": "2 Thessalonians", "I TI-MÔ-THÊ": "1 Timothy",
    "II TI-MÔ-THÊ": "2 Timothy", "TÍT": "Titus", "PHI-LÊ-MÔN": "Philemon",
    "HÊ-BƠ-RƠ": "Hebrews", "GIA-CƠ": "James", "I PHI-E-RƠ": "1 Peter",
    "II PHI-E-RƠ": "2 Peter", "I GIĂNG": "1 John", "II GIĂNG": "2 John",
    "III GIĂNG": "3 John", "GIU-ĐE": "Jude", "KHẢI-HUYỀN": "Revelation"
}

In [ ]:
import re
import requests
from bs4 import BeautifulSoup


def parse_httlvn_chapter_optimized(book_name: str, chapter: int, start_verse: int, end_verse: int, version: str):
    """
    Scrapes a target range of verses for a given version from kinhthanh.httlvn.org.
    Cleans out anchor symbols (⚓) and bypasses structural section titles.
    """
    # 1. Map English book string to platform code automatically
    book_clean = book_name.strip().lower()
    book_code = BOOK_CODE_MAP.get(book_clean)

    if not book_code:
        print(f"❌ Map Error: '{book_name}' not found in BOOK_CODE_MAP database.")
        return {}

    url = f"https://kinhthanh.httlvn.org/doc-kinh-thanh/{book_code}/{chapter}?v={version}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"❌ Error loading {version} from platform. Code: {response.status_code}")
        return {}

    soup = BeautifulSoup(response.content, "lxml")
    content_area = soup.find("div", class_="bible-read")
    if not content_area:
        print(f"❌ Could not find 'bible-read' container for version {version}")
        return {}

    # Strip out all internal title/heading blocks immediately
    for title_div in content_area.find_all(["div", "span"], class_="title"):
        title_div.decompose()

    verse_data = {}
    verse_class_pattern = re.compile(rf"verse\s+{book_code}_{chapter}_\d+")

    for span in content_area.find_all("span", class_=verse_class_pattern):
        sup_tag = span.find("sup")
        if not sup_tag:
            continue

        v_num_text = sup_tag.get_text().strip()
        if not v_num_text.isdigit():
            continue
        v_num = int(v_num_text)

        # BOUNDARY OPTIMIZATION: Ignore this verse completely if it falls outside our targeted range
        if v_num < start_verse or v_num > end_verse:
            continue

        # Remove tooltips and internal anchor text blocks completely
        for anchor in span.find_all("a", class_="data-toggle"):
            anchor.decompose()

        # Extract remaining textual nodes
        raw_text = span.get_text().strip()

        # Remove the leading verse number string prefix
        clean_text = raw_text[len(v_num_text):].strip()

        # CLEANUP UPGRADE: Remove the anchor symbol string explicitly if it persists in text fields
        clean_text = clean_text.replace("⚓", "").strip()

        # Normalize trailing/consecutive whitespace characters
        clean_text = re.sub(r"\s+", " ", clean_text).replace("\xa0", " ")

        if clean_text:
            verse_data[v_num] = clean_text

    return verse_data


def build_bounded_language_cache(book_name: str, chapter: int, start_verse: int, end_verse: int):
    """
    Combines both parsed structures into a unified, range-restricted local memory lookup cache.
    """
    print(f"📡 Bounded Fetch -> Vietnamese (VI1934) for Verses {start_verse}-{end_verse}...")
    vi_verses = parse_httlvn_chapter_optimized(book_name, chapter, start_verse, end_verse, "VI1934")

    print(f"📡 Bounded Fetch -> English (NKJV) for Verses {start_verse}-{end_verse}...")
    en_verses = parse_httlvn_chapter_optimized(book_name, chapter, start_verse, end_verse, "NKJV")

    unified_cache = {}
    all_verse_numbers = set(list(vi_verses.keys()) + list(en_verses.keys()))

    for num in sorted(all_verse_numbers):
        unified_cache[num] = {
            "eng": en_verses.get(num, ""),
            "vie": vi_verses.get(num, "")
        }

    print(f"✅ Bounded alignment complete! Stored {len(unified_cache)} target verses in memory.")
    return unified_cache

## Parsing Pipeline

In [ ]:
def dynamic_bible_pipeline(book_name: str, chapter: int, user_start: int, user_end: int):
    """
    Orchestrates the entire data ingestion. It runs the commentary parser first,
    discovers the true structural boundaries, and dynamically expands the Bible
    cache fetch to match.
    """
    print(f"🎬 Starting Pipeline for {book_name} {chapter}:{user_start}-{user_end}")
    print("------------------------------------------------------------------------")

    # 1. Run the commentary parser with the user's requested scope
    commentary_blocks = parse_enduring_word(book_name, chapter, user_start, user_end)

    if not commentary_blocks:
        print("❌ No commentary blocks extracted. Aborting pipeline.")
        return None, None

    # 2. Find the minimum and maximum verses actually touched by the commentary
    all_touched_verses = []
    for block in commentary_blocks:
        v_start, v_end = block["verse_range"]
        all_touched_verses.extend([v_start, v_end])

    # Determine the true structural bounds computed by the commentary layout
    true_start = min(all_touched_verses)
    true_end = max(all_touched_verses)

    print(f"\n🔄 Boundary Adjustment Verified:")
    print(f"   ↳ User requested: Verses {user_start} - {user_end}")
    print(f"   ↳ Commentary requires: Verses {true_start} - {true_end} (to keep sections complete)")

    # 3. Fetch the Bible reference text using the newly adjusted true boundaries
    optimized_bible_cache = build_bounded_language_cache(book_name, chapter, true_start, true_end)

    print("\n🚀 Pipeline Successfully Synced and Enriched!")
    print("------------------------------------------------------------------------")

    return commentary_blocks, optimized_bible_cache

## Translation Engine

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_translation_engine():
    """Initializes and caches the Qwen tokenizer and model pipeline in GPU memory."""
    model_id = "phuclhp1922/bct_qwen3.5_0.8B_translation_merged_16bit"

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        dtype=torch.bfloat16,
        trust_remote_code=True
    )

    return tokenizer, model

In [ ]:
# Initialize the model and tokenizer directly using your cached layout
tokenizer, model = load_translation_engine()

In [ ]:
import torch

def get_extracted_quote(english_verse: str, vietnamese_verse: str, english_quote: str, tokenizer, model) -> str:
    """
    Prompt 1: Hands the model a single verse sentence and asks it to pull out
    the exact matching Vietnamese equivalent.
    """
    if not english_quote.strip():
        return ""

    messages = [
        {
            "role": "system",
            "content": "You are a bilingual scripture matching assistant. Your job is to extract the exact phrase from the Vietnamese translation that matches the English quote."
        },
        {
            "role": "user",
            "content": (
                f"English Verse: {english_verse}\n"
                f"Vietnamese Verse: {vietnamese_verse}\n"
                f"English Quote to Match: {english_quote}\n\n"
                f"/no_think Extract and output ONLY the corresponding Vietnamese phrase from the Vietnamese Verse."
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(**inputs, do_sample=False, max_new_tokens=64, pad_token_id=tokenizer.eos_token_id)

    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def translate_commentary_with_quote(commentary_text: str, english_quote: str, vietnamese_quote: str, tokenizer, model) -> str:
    """
    Prompt 2: Hands the model the commentary text and pairs the pre-extracted
    Vietnamese phrase with its exact English reference quote block for targeted mapping.
    """
    quote_instruction = ""
    clean_vi = vietnamese_quote.strip()
    clean_en = english_quote.strip().rstrip(":")

    # ─── UPDATED TO EXPLICIT PAIR MAPPING LAYOUT ───
    if clean_vi and clean_en:
        quote_instruction = f"CRITICAL: You must use the exact phrase '{clean_vi}' for '{clean_en}' in this text.\n"
    elif clean_vi:
        quote_instruction = f"CRITICAL: You must use the exact phrase '{clean_vi}' to translate the biblical quote in this text.\n"

    messages = [
        {
            "role": "system",
            "content": "You are a professional theological translator. Your job is to translate English commentary into smooth, traditional Vietnamese biblical study text."
        },
        {
            "role": "user",
            "content": (
                f"/no_think\n"
                f"{quote_instruction}\n"
                f"Translate this commentary text to Vietnamese (keep list markers like 'a.', 'b.', 'i.' intact):\n"
                f"{commentary_text}"
            )
        }
    ]

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    input_token_count = inputs['input_ids'].shape[1]
    dynamic_headroom = max(int(input_token_count * 0.5), 128)

    with torch.inference_mode():
        outputs = model.generate(**inputs, do_sample=False, max_new_tokens=dynamic_headroom, repetition_penalty=1.15, pad_token_id=tokenizer.eos_token_id)

    return tokenizer.decode(outputs[0][input_token_count:], skip_special_tokens=True).strip()

def run_chained_translation_pipeline(commentary_blocks, bible_cache, tokenizer, model):
    """
    Iterates block-by-block through the commentary, executing the two-stage
    extraction and translation chain natively on the GPU.
    """
    print(f"🎬 Starting Chained Pipeline for {len(commentary_blocks)} blocks...\n")
    print("=" * 70)

    for idx, block in enumerate(commentary_blocks, start=1):
        v_start, v_end = block["verse_range"]

        # Default empty strings in case there's no bold quote or missing keys
        extracted_vi_quote = ""
        english_verse_line = ""
        vietnamese_verse_line = ""

        raw_eng_quote = block.get("bold_quote", "").strip()

        # 1. PYTHON LOOKUP: Find the exact verse line from our local cache
        if raw_eng_quote:
            for v_num in range(v_start, v_end + 1):
                if v_num in bible_cache:
                    # Quick python check to see if the phrase belongs to this verse
                    if raw_eng_quote.lower() in bible_cache[v_num]["eng"].lower():
                        english_verse_line = bible_cache[v_num]["eng"]
                        vietnamese_verse_line = bible_cache[v_num]["vie"]
                        break

            # If a strict text substring didn't match perfectly, fallback to the first available verse in the range
            if not english_verse_line and v_start in bible_cache:
                english_verse_line = bible_cache[v_start]["eng"]
                vietnamese_verse_line = bible_cache[v_start]["vie"]

            # 2. PROMPT 1: Ask the model to extract the exact phrase matching our quote
            extracted_vi_quote = get_extracted_quote(
                english_verse=english_verse_line,
                vietnamese_verse=vietnamese_verse_line,
                english_quote=raw_eng_quote,
                tokenizer=tokenizer,
                model=model
            )

        # 3. PROMPT 2: Pass the text and the extracted quote to smooth out the translation
        # Added raw_eng_quote parameter to facilitate explicit string-to-string constraints
        translated_commentary = translate_commentary_with_quote(
            commentary_text=block["full_text"],
            english_quote=raw_eng_quote,
            vietnamese_quote=extracted_vi_quote,
            tokenizer=tokenizer,
            model=model
        )

        # 4. Print results clearly to the console
        print(f"\n--- [BLOCK {idx} / {len(commentary_blocks)} - TYPE: {block['type']}] ---")
        if raw_eng_quote:
            print(f"🇬🇧 English Quote:   {raw_eng_quote}")
            print(f"🇻🇳 Extracted Quote: {extracted_vi_quote}")
        print(f"📝 Translated Text:\n{translated_commentary}")
        print("-" * 50)

## Google Docs API Integration

In [ ]:
import os
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

# 2. Trigger the Google pop-up verification screen
auth.authenticate_user()

# 3. Grab your session credentials
credentials, project_id = google.auth.default()

# 4. Initialize the official Google Docs API engine
docs_service = build('docs', 'v1', credentials=credentials)
print("✨ Google Docs API successfully authorized!")

In [ ]:
# Paste your functions into a cell and run it
def create_blank_study_doc(title: str) -> str:
    body = {'title': title}
    doc_object = docs_service.documents().create(body=body).execute()
    doc_id = doc_object.get('documentId')
    print(f"📄 Document Created: '{title}'")
    print(f"🔗 Access Link: https://docs.google.com/document/d/{doc_id}/edit\n")
    return doc_id

def append_paragraph_to_doc(document_id: str, text_content: str):
    requests = [{
        'insertText': {
            'endOfSegmentLocation': {},
            'text': text_content + "\n\n"
        }
    }]
    docs_service.documents().batchUpdate(documentId=document_id, body={'requests': requests}).execute()

In [ ]:
def append_styled_document_block(document_id: str, block, translated_text: str):
    """
    Appends a commentary block to Google Docs with professional custom typography
    matching your exact layout guidelines.
    """

    # 1. Fetch current document length to find where our new text begins
    doc = docs_service.documents().get(documentId=document_id).execute()
    start_index = doc.get('body').get('content')[-1].get('endIndex') - 1

    # 2. Extract configuration properties from our block
    b_type = block["type"]
    text_content = translated_text.strip() + "\n\n"
    text_length = len(text_content)
    end_index = start_index + text_length

    # 3. Base request to insert the raw characters
    requests = [
        {
            'insertText': {
                'endOfSegmentLocation': {},
                'text': text_content
            }
        },

        # Global Font Family Rule: Enforce Times New Roman across the entire block
        {
            'updateTextStyle': {
                'range': {'startIndex': start_index, 'endIndex': end_index},
                'textStyle': {'weightedFontFamily': {'fontFamily': 'Times New Roman'}},
                'fields': 'weightedFontFamily'
            }
        }
    ]

    # 4. Apply Custom Layout Rules based on structural Block Type
    if b_type == "main_section_header":
        # H3: Red, 20 PT, Bold
        requests.append({
            'updateTextStyle': {
                'range': {'startIndex': start_index, 'endIndex': end_index},
                'textStyle': {
                    'fontSize': {'magnitude': 20, 'unit': 'PT'},
                    'bold': True,
                    'foregroundColor': {'color': {'rgbColor': {'red': 1.0, 'green': 0.0, 'blue': 0.0}}} # Pure Red
                },
                'fields': 'fontSize,bold,foregroundColor'
            }
        })

    elif b_type == "scripture_anchor":
        # H4: Blue, 18 PT, Bold
        requests.append({
            'updateTextStyle': {
                'range': {'startIndex': start_index, 'endIndex': end_index},
                'textStyle': {
                    'fontSize': {'magnitude': 18, 'unit': 'PT'},
                    'bold': True,
                    'foregroundColor': {'color': {'rgbColor': {'red': 0.0, 'green': 0.0, 'blue': 1.0}}} # Pure Blue
                },
                'fields': 'fontSize,bold,foregroundColor'
            }
        })

    else:

        # Regular Body text: 16 PT, Black
        requests.append({
            'updateTextStyle': {
                'range': {'startIndex': start_index, 'endIndex': end_index},
                'textStyle': {'fontSize': {'magnitude': 16, 'unit': 'PT'}},
                'fields': 'fontSize'
            }
        })

        # 5. DYNAMIC INDENTATION CALCULATOR

        indent_level = 0
        stripped_text = text_content.lstrip()

        if stripped_text.startswith(('a.', 'b.', 'c.', 'd.', 'e.')):
            indent_level = 1  # 1 Tab Equivalent
        elif stripped_text.startswith(('i.', 'ii.', 'iii.', 'iv.', 'v.')):
            indent_level = 2  # 2 Tabs Equivalent

        if indent_level > 0:
            # 1 PT = 1/72 inch. Standard Google Doc tab is 0.5 inches (36 PT)
            total_indent_pts = indent_level * 36
            requests.append({
                'updateParagraphStyle': {
                    'range': {'startIndex': start_index, 'endIndex': end_index},
                    'paragraphStyle': {
                        'indentStart': {'magnitude': total_indent_pts, 'unit': 'PT'}
                    },
                    'fields': 'indentStart'
                }
            })

    # 6. Ship the styling requests to the cloud
    docs_service.documents().batchUpdate(documentId=document_id, body={'requests': requests}).execute()

In [ ]:
def run_live_production_pipeline(commentary_blocks, bible_cache, target_doc_id, tokenizer, model):
    """
    Iterates block-by-block, runs local GPU extraction and translation models,
    and streams the resulting text directly into a styled Google Doc.
    """
    print(f"🎬 Starting LIVE production processing for {len(commentary_blocks)} blocks...")
    print(f"📄 Target Google Doc ID: {target_doc_id}\n")
    print("=" * 70)

    for idx, block in enumerate(commentary_blocks, start=1):
        v_start, v_end = block["verse_range"]

        extracted_vi_quote = ""
        english_verse_line = ""
        vietnamese_verse_line = ""

        # 1. PYTHON LOOKUP: Find the exact verse line from your local cache
        if block["bold_quote"]:
            for v_num in range(v_start, v_end + 1):
                if v_num in bible_cache:
                    if block["bold_quote"].lower() in bible_cache[v_num]["eng"].lower():
                        english_verse_line = bible_cache[v_num]["eng"]
                        vietnamese_verse_line = bible_cache[v_num]["vie"]
                        break

            if not english_verse_line and v_start in bible_cache:
                english_verse_line = bible_cache[v_start]["eng"]
                vietnamese_verse_line = bible_cache[v_start]["vie"]

            # 2. PROMPT 1: Direct local GPU phrase extraction
            extracted_vi_quote = get_extracted_quote(
                english_verse=english_verse_line,
                vietnamese_verse=vietnamese_verse_line,
                english_quote=block["bold_quote"],
                tokenizer=tokenizer,
                model=model
            )
            print(f"🔍 Khối {idx} - Cụm từ khớp: \"{extracted_vi_quote}\"")

        # 3. PROMPT 2: Direct local GPU Contextual Translation
        translated_commentary = translate_commentary_with_quote(
            commentary_text=block["full_text"],
            english_quote=block["bold_quote"],
            vietnamese_quote=extracted_vi_quote,
            tokenizer=tokenizer,
            model=model
        )

        # 4. PUSH & STYLE LIVE ON GOOGLE DOCS
        # Drops the text straight into your working custom typography layout engine
        append_styled_document_block(target_doc_id, block, translated_commentary)
        print(f"🎨 Styled & Pushed Block {idx}/{len(commentary_blocks)} to Google Docs.\n")

    print("\n🎉 Done! Your entire live chapter is translated, structured, and styled on Google Drive.")

In [ ]:
# 1. Create your clean web document canvas
production_doc_id = create_blank_study_doc("LESSON 30")

commentary_blocks, bible_cache = dynamic_bible_pipeline("Hebrews", 12, 1, 13)

# 2. Process your blocks natively through the integrated chain
run_live_production_pipeline(
    commentary_blocks=commentary_blocks,
    bible_cache=bible_cache,
    target_doc_id=production_doc_id,
    tokenizer=tokenizer,
    model=model
)